# CSC 2301 · Week 10 — Data tables with pandas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W10_pandas_drillholes.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

In weeks 1–7 you read files with the `csv` module and wrote loops to group and summarise. **pandas** does
the same work on whole tables in a few lines, and is the tool used in industry for tabular data.
The data are the week 1 assays and the week 5 collars: you will redo, in a few lines, what took a
whole lab then.

**What you will practise:** `read_csv` · `head`, `info`, `describe` · selecting columns and rows ·
new columns · Boolean selection · `groupby` · `merge` · `sort_values` · `to_csv`.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('data/drillholes_assays.csv', 'drillholes_assays.csv'), ('data/collars.csv', 'collars.csv'), ('tests/w10/q1.py', 'tests/q1.py'), ('tests/w10/q2.py', 'tests/q2.py'), ('tests/w10/q3.py', 'tests/q3.py'), ('tests/w10/q4.py', 'tests/q4.py'), ('tests/w10/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. A DataFrame is a table

Rows are records, columns are variables. Each column is a NumPy array with a name.

In [ ]:
import pandas as pd

df = pd.read_csv("drillholes_assays.csv")
df.head()

In [ ]:
df.info()
df.describe()

## 2. Selecting

In [ ]:
df["au_ppb"].head(3)                 # one column (a Series)
df[["hole_id", "au_ppb"]].head(3)    # several columns (a DataFrame)
df[df["lithology"] == "quartz vein"] # rows that meet a condition

In [ ]:
df.loc[df["au_ppb"] > 5000, ["hole_id", "from_m", "au_ppb"]]   # rows AND columns

## 3. Grouping: split, apply, combine

`groupby` splits the table by a key, applies a calculation to each group and combines the results.

In [ ]:
df.groupby("lithology")["au_ppb"].agg(["count", "mean", "max"]).round(0)

---
## Exercises

### Q1. Load the tables

Read `drillholes_assays.csv` into `assays` and `collars.csv` into `collars`. Store the number of rows of
`assays` in `n_rows` and the number of different holes in `n_holes`.

In [ ]:
assays = ...
collars = ...
n_rows = ...
n_holes = ...
n_rows, n_holes

In [ ]:
grader.check("q1")

### Q2. New columns

Add to `assays` the columns `length_m` (to_m − from_m) and `au_gpt` (au_ppb / 1000). No loop.

In [ ]:
...
assays.head()

In [ ]:
grader.check("q2")

### Q3. Selecting rows

Store in `high_grade` the intervals with `au_gpt >= 2`, and in `granite_mean` the mean `au_gpt` of the granite
intervals.

In [ ]:
high_grade = ...
granite_mean = ...
len(high_grade), granite_mean

In [ ]:
grader.check("q3")

### Q4. Weighted mean per hole with `groupby`

Store in `weighted_by_hole` a Series: hole_id → length-weighted mean grade in g/t. *Hint:* add a column
`metal = au_gpt × length_m`, sum `metal` and `length_m` per hole, then divide.
Compare with your week 1 answer to Q5.

In [ ]:
weighted_by_hole = ...
weighted_by_hole.round(3)

In [ ]:
grader.check("q4")

### Q5. Merge, sort and save

Build `summary`: one row per hole with the columns `hole_id`, `easting_m`, `northing_m` and `mean_gpt`
(from `weighted_by_hole`), sorted from the richest hole to the poorest. Save it as `hole_summary.csv`
without the index.

*Hint:* `weighted_by_hole.rename("mean_gpt").reset_index()` turns the Series into a table you can
`merge` with `collars` on `hole_id`.

In [ ]:
summary = ...
summary

In [ ]:
grader.check("q5")

### Going further (not checked)

* `pd.read_csv` can read the week 4 raw report too: try `sep=";"`, `skiprows`, `decimal=","`. What still has to
  be cleaned by hand?
* Plot `summary` on a map, the point size proportional to `mean_gpt`: `summary.plot.scatter(...)`.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()